# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alsa-mirza/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: One row represents one search query for one page on one date.

Time window: I will use data from the month 2026-03 because it is a middle month and avoids using the final month as the evaluation period.

In [10]:
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
print("Token loaded successfully!" if HF_TOKEN else "Token not found!")


Token loaded successfully!


In [11]:
!pip -q install datasets duckdb huggingface_hub pyarrow

In [12]:
from datasets import load_dataset

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_query_90d",
    split="train",
    token=HF_TOKEN
)
print(dataset)

Dataset({
    features: ['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share'],
    num_rows: 2414248
})


In [13]:
df = dataset.to_pandas()

print(df.shape)
df.head()

(2414248, 21)


,client_hash_id,content_hash_id,query_hash_id,query_char_count,query_token_count,window_start,window_end,impressions_90d,clicks_90d,impressions_last30,...,impressions_prev30,clicks_prev30,avg_position_90d,avg_position_last30,avg_position_prev30,content_total_impressions_90d,content_visible_query_count,rare_query_count,rare_impressions_share,anonymized_impressions_share
0,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_58b1b001f839d699,17,3,2026-04-02,2026-06-30,11,0,0,...,11,0,10.818182,NaN,10.818182,1466,14,32,0.043656,0.725102
1,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_922b8eca2a24cd34,34,7,2026-04-02,2026-06-30,13,0,0,...,1,0,1.769231,NaN,11.000000,1466,14,32,0.043656,0.725102
2,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_9f0c36a6ae2a6a99,16,2,2026-04-02,2026-06-30,16,0,11,...,5,0,23.562500,24.272727,22.000000,1466,14,32,0.043656,0.725102
3,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_a032820b5467e996,24,4,2026-04-02,2026-06-30,55,0,1,...,1,0,2.200000,13.000000,0.000000,1466,14,32,0.043656,0.725102
4,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_ba1a2f131961c5da,18,3,2026-04-02,2026-06-30,14,0,0,...,0,0,3.428571,NaN,NaN,1466,14,32,0.043656,0.725102


In [14]:
df.columns.tolist()

['client_hash_id',
 'content_hash_id',
 'query_hash_id',
 'query_char_count',
 'query_token_count',
 'window_start',
 'window_end',
 'impressions_90d',
 'clicks_90d',
 'impressions_last30',
 'clicks_last30',
 'impressions_prev30',
 'clicks_prev30',
 'avg_position_90d',
 'avg_position_last30',
 'avg_position_prev30',
 'content_total_impressions_90d',
 'content_visible_query_count',
 'rare_query_count',
 'rare_impressions_share',
 'anonymized_impressions_share']

In [15]:
df.head()

,client_hash_id,content_hash_id,query_hash_id,query_char_count,query_token_count,window_start,window_end,impressions_90d,clicks_90d,impressions_last30,...,impressions_prev30,clicks_prev30,avg_position_90d,avg_position_last30,avg_position_prev30,content_total_impressions_90d,content_visible_query_count,rare_query_count,rare_impressions_share,anonymized_impressions_share
0,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_58b1b001f839d699,17,3,2026-04-02,2026-06-30,11,0,0,...,11,0,10.818182,NaN,10.818182,1466,14,32,0.043656,0.725102
1,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_922b8eca2a24cd34,34,7,2026-04-02,2026-06-30,13,0,0,...,1,0,1.769231,NaN,11.000000,1466,14,32,0.043656,0.725102
2,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_9f0c36a6ae2a6a99,16,2,2026-04-02,2026-06-30,16,0,11,...,5,0,23.562500,24.272727,22.000000,1466,14,32,0.043656,0.725102
3,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_a032820b5467e996,24,4,2026-04-02,2026-06-30,55,0,1,...,1,0,2.200000,13.000000,0.000000,1466,14,32,0.043656,0.725102
4,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_ba1a2f131961c5da,18,3,2026-04-02,2026-06-30,14,0,0,...,0,0,3.428571,NaN,NaN,1466,14,32,0.043656,0.725102


In [16]:
df[['window_start','window_end']].head(10)

,window_start,window_end
0,2026-04-02,2026-06-30
1,2026-04-02,2026-06-30
2,2026-04-02,2026-06-30
3,2026-04-02,2026-06-30
4,2026-04-02,2026-06-30
5,2026-04-02,2026-06-30
6,2026-04-02,2026-06-30
7,2026-04-02,2026-06-30
8,2026-04-02,2026-06-30
9,2026-04-02,2026-06-30


## 2. Fields: feature / label / context / excluded

Feature:
- impressions_last30
- clicks_last30
- avg_position_last30
- query_char_count
- query_token_count

Label:
- clicks_90d

Context:
- client_hash_id
- content_hash_id
- window_start
- window_end

Excluded:
- clicks_90d (as feature because it causes data leakage)

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

Verification Results:
1. Verified the total number of rows in the dataset.
2. Verified the available date range using window_start and window_end.
3. Verified that the selected fields do not contain missing values.

In [18]:
import pandas as pd

# Query 1: Total rows
print("Total Rows:", len(df))

# Query 2: Date range
print("\nWindow Start:", df["window_start"].min())
print("Window End:", df["window_end"].max())

# Query 3: Missing values
print("\nMissing Values:")
print(df.isnull().sum())


Total Rows: 2414248

Window Start: 2026-04-02
Window End: 2026-06-30

Missing Values:
client_hash_id                        0
content_hash_id                       0
query_hash_id                         0
query_char_count                      0
query_token_count                     0
window_start                          0
window_end                            0
impressions_90d                       0
clicks_90d                            0
impressions_last30                    0
clicks_last30                         0
impressions_prev30                    0
clicks_prev30                         0
avg_position_90d                      0
avg_position_last30              530758
avg_position_prev30              329420
content_total_impressions_90d         0
content_visible_query_count           0
rare_query_count                      0
rare_impressions_share                0
anonymized_impressions_share          0
dtype: int64


## 4. Data limits

Data Limitation:
This dataset only contains historical search performance data. It does not include user behavior outside search engines, seasonal events, marketing campaigns, or future information. Therefore, predictions are limited to patterns available in the historical search data.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.